In [71]:
import pandas as pd
from utils.funcs import get_variable_names, rename_columns, processed_diabetes_data, demographic_data
from functools import reduce

In [72]:
def read_file(filename=''):

    dataset_path='raw_datasets'
    df=pd.read_sas(f'{dataset_path}/{filename}.XPT', format='xport')
    mapping=get_variable_names()
    df=rename_columns(df, mapping)  
    
    return df

In [73]:
df_diabetes=processed_diabetes_data()  #Taken data of people which give answer as yes or no
df_demographic=demographic_data()
df_audio=read_file('audiometry')
df_blood_pressure=read_file('blood_pressure_cholesterol')
df_general_health=read_file('hospital_utilization_access_to_care')
df_weight=read_file('weight_history')
df_occupation=read_file('occupation')

In [74]:
dataframes = [df_diabetes,df_demographic, df_audio, df_blood_pressure, df_general_health, df_weight, df_occupation]
df = reduce(lambda left, right: pd.merge(left, right, on='sequence_no', how='inner'), dataframes)
columns_to_select = ['sequence_no', 'EverTold_Diabetes', 'gender', 'age','weight','HearingStatus_NoAid','EverTold_Hypertension','EverTold_HighCholesterol','GeneralHealth_Status','CurrentHeight','CurrentWeight','WorkExperience_LastWeek','WeightOneYearAgo']
df=df[columns_to_select]
df['bmi']=df['CurrentWeight']/(df['CurrentHeight']**2)

In [ ]:
print("Number of rows having NAN values:",df.isnull().any(axis=1).sum())

In [76]:
df = df.dropna()

In [77]:
X = df.drop(['EverTold_Diabetes','sequence_no'], axis=1)  
y = df["EverTold_Diabetes"] 
y=y.map({1.0: 1, 2.0: 0})

In [58]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_selection import SelectKBest, f_classif

In [44]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
importances = rf.feature_importances_

In [45]:
selector = SelectKBest(score_func=f_classif, k=10)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

In [46]:
rf_selected = RandomForestClassifier()
rf_selected.fit(X_train_selected, y_train)
y_pred = rf_selected.predict(X_test_selected)

In [ ]:
print(classification_report(y_test, y_pred))


In [ ]:
importances

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

selector = SelectKBest(score_func=f_classif, k=10)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

log_reg = LogisticRegression( max_iter=100)

# Train Logistic Regression on selected features
log_reg.fit(X_train_selected, y_train)
y_pred = log_reg.predict(X_test_selected)

# Evaluate again
print("Accuracy (selected features):", accuracy_score(y_test, y_pred))
print("\nClassification Report (selected features):")
print(classification_report(y_test, y_pred))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get feature names and coefficients
feature_names = X.columns  # Replace with actual feature column names
coefficients = log_reg.coef_[0]  # Coefficients for each feature

# Sort features by importance
sorted_indices = np.argsort(np.abs(coefficients))[::-1]  # Sort by absolute value of coefficients
sorted_feature_names = feature_names[sorted_indices]
sorted_coefficients = coefficients[sorted_indices]

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(sorted_feature_names, sorted_coefficients, color="skyblue")
plt.xlabel("Coefficient Value")
plt.title("Feature Importance (Logistic Regression)")
plt.gca().invert_yaxis()  # Invert y-axis for readability
plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],  # Use 'l1' or 'elasticnet' if your solver supports it
    'solver': ['lbfgs', 'saga']  # Ensure compatibility with penalty
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='roc_auc',  # Optimize for AUC-ROC
    n_jobs=-1
)

# Fit the model
grid_search.fit(X_train_scaled, y_train)

# Best hyperparameters and model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
print("Best Hyperparameters:", best_params)

# Evaluate the best model
y_pred_best = best_model.predict(X_test_scaled)
y_pred_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]
print("AUC-ROC (Best Model):", roc_auc_score(y_test, y_pred_proba_best))
print("Classification Report (Best Model):")
print(classification_report(y_test, y_pred_best))


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix

# Step 1: Load your dataset
# Replace 'your_diabetes_dataset.csv' with you

# Step 2: Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Step 3: Preprocess the features (scale numerical features)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 4: Train an SVM model
svm = SVC(kernel='rbf', probability=True, random_state=42,class_weight='balanced')  # Using RBF kernel
svm.fit(X_train_scaled, y_train)

# Step 5: Make predictions
y_pred = svm.predict(X_test_scaled)
y_pred_proba = svm.predict_proba(X_test_scaled)[:, 1]  # Probabilities for positive class

# Step 6: Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
# Define the parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': [0.01, 0.1, 1, 'scale']  # 'scale' adjusts gamma automatically
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='roc_auc',  # Optimize for AUC-ROC
    n_jobs=-1
)

# Fit the model
grid_search.fit(X_train_scaled, y_train)

# Best hyperparameters and model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
print("Best Hyperparameters:", best_params)

# Evaluate the best model
y_pred_best = best_model.predict(X_test_scaled)
y_pred_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]
print("AUC-ROC (Best Model):", roc_auc_score(y_test, y_pred_proba_best))
print("Classification Report (Best Model):")
print(classification_report(y_test, y_pred_best))


In [11]:
from imblearn.combine import SMOTEENN
smote_enn = SMOTEENN(random_state=42)
X_train_resampled, y_train_resampled = smote_enn.fit_resample(X_train, y_train)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

# Step 4: Train an SVM model
svm = SVC(kernel='rbf', probability=True, random_state=42,class_weight='balanced')  # Using RBF kernel
svm.fit(X_train_scaled, y_train_resampled)

# Step 5: Make predictions
y_pred = svm.predict(X_test_scaled)
y_pred_proba = svm.predict_proba(X_test_scaled)[:, 1]  # Probabilities for positive class

# Step 6: Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train_scaled, y_train_resampled)

y_pred = rf.predict(X_test_scaled)
print("Classification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
rf.fit(X_train_scaled, y_train)

y_pred = rf.predict(X_test_scaled)
print("Classification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
!pip install lightgbm

In [ ]:
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score

# Create LightGBM datasets
train_data = lgb.Dataset(X_train_scaled, label=y_train_resampled)
test_data = lgb.Dataset(X_test_scaled, label=y_test, reference=train_data)

# Define parameters
params = {
    'objective': 'binary',  # Binary classification
    'boosting_type': 'gbdt',  # Gradient Boosting Decision Trees
    'metric': 'auc',  # Evaluation metric
    'learning_rate': 0.05,  # Step size
    'num_leaves': 31,  # Number of leaves in one tree
    'max_depth': -1,  # No limit for tree depth
    'scale_pos_weight': len(y_train[y_train == 1.0]) / len(y_train[y_train == 2.0]),  # Handle class imbalance
    'feature_fraction': 0.8,  # Randomly select 80% of features for training
    'bagging_fraction': 0.8,  # Randomly select 80% of data for training
    'bagging_freq': 5,  # Perform bagging every 5 iterations
    'seed': 42
}

# Train the model
lgb_model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],  # Pass training and validation datasets
    valid_names=['train', 'valid'],
    num_boost_round=1000,  # Maximum number of boosting rounds
)

# Predictions
y_pred_proba = lgb_model.predict(X_test_scaled, num_iteration=lgb_model.best_iteration)
y_pred = (y_pred_proba >= 0.5).astype(int)

# Evaluate the model
print("AUC-ROC:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [60]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define functions for training and evaluating SVM
def train_and_evaluate_svm(X_train, y_train, X_test, y_test):
    # Train an SVM
    svm = SVC(kernel='rbf', probability=True, random_state=42)
    svm.fit(X_train, y_train)

    # Make predictions
    y_pred = svm.predict(X_test)
    y_pred_proba = svm.predict_proba(X_test)[:, 1]

    # Evaluate performance
    print("AUC-ROC:", roc_auc_score(y_test, y_pred_proba))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# Original data (imbalanced)
print("### Original Data ###")
train_and_evaluate_svm(X_train_scaled, y_train, X_test_scaled, y_test)

# SMOTE (Oversampling)
print("\n### SMOTE ###")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
train_and_evaluate_svm(X_train_smote, y_train_smote, X_test_scaled, y_test)

# RUS (Under-sampling)
print("\n### RUS ###")
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_scaled, y_train)
train_and_evaluate_svm(X_train_rus, y_train_rus, X_test_scaled, y_test)

# SMOTE + RUS (Hybrid)
print("\n### SMOTE + RUS ###")
smote_rus = SMOTETomek(random_state=42)
X_train_smote_rus, y_train_smote_rus = smote_rus.fit_resample(X_train_scaled, y_train)
train_and_evaluate_svm(X_train_smote_rus, y_train_smote_rus, X_test_scaled, y_test)


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
import numpy as np
# Map 1.0 to 0 and 2.0 to 1

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define a function for K-Fold Cross-Validation
def evaluate_with_kfold(X, y, sampler=None, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # Define the SVM model
    svm = SVC(kernel='rbf', probability=True, random_state=42)
    
    # Create a pipeline if a sampler is provided
    if sampler:
        pipeline = Pipeline(steps=[('sampler', sampler), ('svm', svm)])
    else:
        pipeline = Pipeline(steps=[('svm', svm)])
    
    # Perform cross-validation and get predictions
    y_pred_proba = cross_val_predict(
        pipeline, X, y, cv=skf, method='predict_proba'
    )
    y_pred = (y_pred_proba[:, 1] >= 0.5).astype(int)
    
    # Evaluate the model
    auc_roc = roc_auc_score(y, y_pred_proba[:, 1])
    print("AUC-ROC:", auc_roc)
    print("\nClassification Report:")
    print(classification_report(y, y_pred))
    return auc_roc

# Evaluate Original Data (No Resampling)
print("### Original Data ###")
auc_original = evaluate_with_kfold(X_scaled, y)

# Evaluate SMOTE
print("\n### SMOTE ###")
smote = SMOTE(random_state=42)
auc_smote = evaluate_with_kfold(X_scaled, y, sampler=smote)

# Evaluate RUS
print("\n### RUS ###")
rus = RandomUnderSampler(random_state=42)
auc_rus = evaluate_with_kfold(X_scaled, y, sampler=rus)

# Evaluate SMOTE + RUS
print("\n### SMOTE + RUS ###")
smote_rus = SMOTETomek(random_state=42)
auc_smote_rus = evaluate_with_kfold(X_scaled, y, sampler=smote_rus)

# Summary of Results
print("\n### Summary of Results ###")
print(f"Original Data AUC-ROC: {auc_original:.4f}")
print(f"SMOTE AUC-ROC: {auc_smote:.4f}")
print(f"RUS AUC-ROC: {auc_rus:.4f}")
print(f"SMOTE + RUS AUC-ROC: {auc_smote_rus:.4f}")


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
import numpy as np
# Map 1.0 to 0 and 2.0 to 1

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define a function for K-Fold Cross-Validation
def evaluate_with_kfold(X, y, sampler=None, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # Define the SVM model
    svm = SVC(kernel='rbf', probability=True,class_weight='balanced' ,random_state=42)
    
    # Create a pipeline if a sampler is provided
    if sampler:
        pipeline = Pipeline(steps=[('sampler', sampler), ('svm', svm)])
    else:
        pipeline = Pipeline(steps=[('svm', svm)])
    
    # Perform cross-validation and get predictions
    y_pred_proba = cross_val_predict(
        pipeline, X, y, cv=skf, method='predict_proba'
    )
    y_pred = (y_pred_proba[:, 1] >= 0.5).astype(int)
    
    # Evaluate the model
    auc_roc = roc_auc_score(y, y_pred_proba[:, 1])
    print("AUC-ROC:", auc_roc)
    print("\nClassification Report:")
    print(classification_report(y, y_pred))
    return auc_roc

# Evaluate Original Data (No Resampling)
print("### Original Data ###")
auc_original = evaluate_with_kfold(X_scaled, y)

# Evaluate SMOTE
print("\n### SMOTE ###")
smote = SMOTE(random_state=42)
auc_smote = evaluate_with_kfold(X_scaled, y, sampler=smote)

# Evaluate RUS
print("\n### RUS ###")
rus = RandomUnderSampler(random_state=42)
auc_rus = evaluate_with_kfold(X_scaled, y, sampler=rus)

# Evaluate SMOTE + RUS
print("\n### SMOTE + RUS ###")
smote_rus = SMOTETomek(random_state=42)
auc_smote_rus = evaluate_with_kfold(X_scaled, y, sampler=smote_rus)

# Summary of Results
print("\n### Summary of Results ###")
print(f"Original Data AUC-ROC: {auc_original:.4f}")
print(f"SMOTE AUC-ROC: {auc_smote:.4f}")
print(f"RUS AUC-ROC: {auc_rus:.4f}")
print(f"SMOTE + RUS AUC-ROC: {auc_smote_rus:.4f}")


In [ ]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# Assume X and y are your features and target labels
# Map labels if needed

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Define the Optuna objective function
def objective(trial):
    # Define hyperparameter search space
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "dart"]),
        "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.1),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "max_depth": trial.suggest_int("max_depth", -1, 20),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
        "feature_fraction": trial.suggest_uniform("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_uniform("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_loguniform("lambda_l1", 1e-8, 10.0),
        "lambda_l2": trial.suggest_loguniform("lambda_l2", 1e-8, 10.0),
        "min_gain_to_split": trial.suggest_uniform("min_gain_to_split", 0.0, 1.0),  # Fixed
    }


    # Create LightGBM datasets
    train_data = lgb.Dataset(X_train_scaled, label=y_train)
    val_data = lgb.Dataset(X_val_scaled, label=y_val, reference=train_data)

    # Train LightGBM model
    gbm = lgb.train(
        params,
        train_data,
        valid_sets=[val_data],
        valid_names=["valid"],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50),lgb.log_evaluation(period=50)]
    )

    # Predict probabilities and evaluate AUC-ROC
    y_pred = gbm.predict(X_val_scaled, num_iteration=gbm.best_iteration)
    auc = roc_auc_score(y_val, y_pred)
    return auc

# Create and optimize an Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)  # Specify the number of trials

# Print the best parameters
print("Best Parameters:")
print(study.best_params)

# Train and evaluate the model with the best parameters
best_params = study.best_params
best_params.update({"objective": "binary", "metric": "auc"})

train_data = lgb.Dataset(X_train_scaled, label=y_train)
val_data = lgb.Dataset(X_val_scaled, label=y_val, reference=train_data)

best_model = lgb.train(
    best_params,
    train_data,
    valid_sets=[val_data],
    valid_names=["valid"],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(stopping_rounds=50),lgb.log_evaluation(period=50)]
)

# Evaluate the best model
y_pred = best_model.predict(X_val_scaled, num_iteration=best_model.best_iteration)
auc_best = roc_auc_score(y_val, y_pred)
print(f"Best AUC-ROC: {auc_best:.4f}")
